# dARK Core Minter API - End-to-End Functional Test Notebook

This notebook validates the main API capabilities and edge cases:

- Health and worker status checks
- Authority setup and authorization endpoints
- ARK reserve/get/update/delete lifecycle
- JSON and XML metadata handling
- Validation and ownership error scenarios
- Batch reserve scenarios
- Concurrent minting uniqueness checks
- Optional worker publish/update flow (`DRAFT -> PUBLISHED`, `PUBLISHED -> UPDATE -> PUBLISHED`)
- Optional chain-import path (`PUT` over ARK existing only on-chain)

Run this notebook top-to-bottom.


In [ ]:
import json
import os
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pprint import pprint

import requests

# -----------------------------------------------------------------------------
# Runtime configuration
# -----------------------------------------------------------------------------
MINTER_BASE_URL = os.getenv("MINTER_BASE_URL", "http://localhost:8001").rstrip("/")
MINTER_API_V1 = f"{MINTER_BASE_URL}/api/v1"
ADMIN_API_URL = os.getenv("ADMIN_API_URL", "http://localhost:8000/api/v1/admin").rstrip("/")

AUTHORITY_ID = os.getenv("AUTHORITY_ID", f"test-authority-{int(time.time())}")
NAAN = os.getenv("NAAN", "12345")

REGISTER_AUTHORITY = os.getenv("REGISTER_AUTHORITY", "true").lower() in {"1", "true", "yes", "y"}
EXISTING_CHAIN_ARK = os.getenv("EXISTING_CHAIN_ARK", "").strip() or None

POLL_INTERVAL_SECONDS = float(os.getenv("POLL_INTERVAL_SECONDS", "2"))
POLL_TIMEOUT_SECONDS = int(os.getenv("POLL_TIMEOUT_SECONDS", "90"))

print("MINTER_BASE_URL:", MINTER_BASE_URL)
print("MINTER_API_V1:", MINTER_API_V1)
print("ADMIN_API_URL:", ADMIN_API_URL)
print("AUTHORITY_ID:", AUTHORITY_ID)
print("NAAN:", NAAN)
print("REGISTER_AUTHORITY:", REGISTER_AUTHORITY)
print("EXISTING_CHAIN_ARK:", EXISTING_CHAIN_ARK)
print("POLL_TIMEOUT_SECONDS:", POLL_TIMEOUT_SECONDS)

# -----------------------------------------------------------------------------
# Test result tracking
# -----------------------------------------------------------------------------
RESULTS = []
CONTEXT = {}


def record(name: str, status: str, detail: str = ""):
    status = status.upper()
    if status not in {"PASS", "FAIL", "SKIP"}:
        raise ValueError(f"Invalid status: {status}")
    RESULTS.append({"name": name, "status": status, "detail": detail})
    print(f"[{status}] {name}")
    if detail:
        print("       ", detail)


def check(name: str, condition: bool, ok_detail: str = "", fail_detail: str = ""):
    if condition:
        record(name, "PASS", ok_detail)
    else:
        record(name, "FAIL", fail_detail or "Condition evaluated to False")


def skip(name: str, detail: str = ""):
    record(name, "SKIP", detail)


def summarize_results():
    passed = sum(1 for r in RESULTS if r["status"] == "PASS")
    failed = sum(1 for r in RESULTS if r["status"] == "FAIL")
    skipped = sum(1 for r in RESULTS if r["status"] == "SKIP")
    total = len(RESULTS)

    print("\n=== TEST SUMMARY ===")
    print(f"Total checks : {total}")
    print(f"Passed       : {passed}")
    print(f"Failed       : {failed}")
    print(f"Skipped      : {skipped}")

    if failed:
        print("\nFailed checks:")
        for row in RESULTS:
            if row["status"] == "FAIL":
                print(f"- {row['name']}: {row['detail']}")

    return {"total": total, "passed": passed, "failed": failed, "skipped": skipped}


def _decode_body(resp: requests.Response):
    ctype = (resp.headers.get("content-type") or "").lower()
    if "application/json" in ctype:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        return resp.json()
    except Exception:
        return resp.text


def call_api(name: str, method: str, path: str, expected, base: str = MINTER_API_V1, timeout: int = 30, **kwargs):
    url = f"{base}{path}"
    exp = expected if isinstance(expected, (list, tuple, set)) else [expected]

    try:
        resp = requests.request(method=method.upper(), url=url, timeout=timeout, **kwargs)
    except Exception as exc:
        record(name, "FAIL", f"Request exception against {url}: {exc}")
        return None, None

    body = _decode_body(resp)
    ok = resp.status_code in exp
    detail = f"expected={list(exp)} got={resp.status_code} method={method.upper()} url={url}"
    record(name, "PASS" if ok else "FAIL", detail)

    print(f"Status: {resp.status_code}")
    if isinstance(body, dict):
        pprint(body)
    else:
        text = str(body)
        if len(text) > 1000:
            text = text[:1000] + "..."
        print(text)

    return resp, body


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def json_metadata(title: str, version: int = 1):
    return json.dumps(
        {
            "title": title,
            "version": version,
            "source": "notebook",
            "timestamp": now_iso(),
            "description": "Integration metadata payload",
        },
        ensure_ascii=True,
    )


def xml_metadata(title: str):
    return f'''<?xml version="1.0" encoding="UTF-8"?>
<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/"
           xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:title>{title}</dc:title>
  <dc:creator>Notebook Integration Test</dc:creator>
  <dc:date>{now_iso()}</dc:date>
  <dc:description>XML metadata payload for integration test</dc:description>
</oai_dc:dc>'''


def wait_for_state(ark: str, expected_state: str, timeout_seconds: int = POLL_TIMEOUT_SECONDS):
    started = time.time()
    while True:
        resp, body = call_api(
            name=f"Poll ARK state={expected_state}",
            method="GET",
            path=f"/arks/{ark}",
            expected=[200, 404],
        )
        if resp is not None and resp.status_code == 200 and isinstance(body, dict):
            if body.get("state") == expected_state:
                return True, body

        if time.time() - started >= timeout_seconds:
            return False, body

        time.sleep(POLL_INTERVAL_SECONDS)


## 1. Service Smoke Checks

This section validates connectivity to API endpoints before deeper lifecycle tests.


In [ ]:
# Health endpoint (root level)
health_resp, health_body = call_api(
    name="Health endpoint reachable",
    method="GET",
    path="/health",
    expected=[200, 503],
    base=MINTER_BASE_URL,
)

# Worker status endpoint (API v1)
worker_resp, worker_body = call_api(
    name="Worker status endpoint reachable",
    method="GET",
    path="/worker/status",
    expected=200,
)

worker_running = bool(isinstance(worker_body, dict) and worker_body.get("running"))
check(
    name="Worker status has expected shape",
    condition=isinstance(worker_body, dict) and "status" in worker_body,
    ok_detail=f"worker_running={worker_running}",
    fail_detail="/worker/status did not return expected JSON",
)

CONTEXT["worker_running"] = worker_running


## 2. Authority Setup and Authorization Endpoints

If `REGISTER_AUTHORITY=true`, this notebook attempts to register a fresh authority using the Admin API.
Then it validates authority-related Minter API endpoints.


In [ ]:
def _get_authz():
    try:
        r = requests.get(f"{MINTER_API_V1}/authority/{AUTHORITY_ID}/authorized/{NAAN}", timeout=30)
        return r.status_code, _decode_body(r)
    except Exception as exc:
        return None, str(exc)


def _wait_authorized(timeout_seconds: int = 90):
    started = time.time()
    last = None
    while time.time() - started <= timeout_seconds:
        status, body = _get_authz()
        last = (status, body)
        if status == 200 and isinstance(body, dict) and body.get("authorized") is True:
            return True, last
        time.sleep(2)
    return False, last


if REGISTER_AUTHORITY:
    payload = {
        "uuid": AUTHORITY_ID,
        "naans": [NAAN],
        "fund_amount_eth": 0.05,
    }

    register_ok = False
    register_detail = ""
    for attempt in range(1, 3):
        try:
            resp = requests.post(f"{ADMIN_API_URL}/authority", json=payload, timeout=180)
            body = _decode_body(resp)
            if resp.status_code in {200, 201, 409}:
                register_ok = True
                register_detail = f"status={resp.status_code} attempt={attempt}"
                break
            register_detail = f"status={resp.status_code} body={body} attempt={attempt}"
        except Exception as exc:
            register_detail = f"attempt={attempt} exception={exc}"
        time.sleep(2)

    record("Admin register authority", "PASS" if register_ok else "FAIL", register_detail)

    # If registration did not result in immediate authorization, enforce NAAN authorization.
    authorized_now, last_auth = _wait_authorized(timeout_seconds=30)
    if not authorized_now:
        try:
            auth_resp = requests.post(
                f"{ADMIN_API_URL}/authority/{AUTHORITY_ID}/authorize-naan",
                json={"naan": NAAN},
                timeout=120,
            )
            auth_body = _decode_body(auth_resp)
            record(
                "Admin authorize NAAN fallback",
                "PASS" if auth_resp.status_code == 200 else "FAIL",
                f"status={auth_resp.status_code} body={auth_body}",
            )
        except Exception as exc:
            record("Admin authorize NAAN fallback", "FAIL", str(exc))
else:
    skip("Admin register authority", "REGISTER_AUTHORITY=false")

# Validate authority endpoints on minter API
_, auth_body = call_api(
    name="Authority details",
    method="GET",
    path=f"/authority/{AUTHORITY_ID}",
    expected=200,
)

_, naans_body = call_api(
    name="Authority NAAN list",
    method="GET",
    path=f"/authority/{AUTHORITY_ID}/naans",
    expected=200,
)

authorized_ok, last_auth = _wait_authorized(timeout_seconds=90)
record(
    "Authority authorization check",
    "PASS" if authorized_ok else "FAIL",
    f"last_auth={last_auth}",
)

if authorized_ok:
    check(
        name="Authority is authorized for NAAN",
        condition=True,
        ok_detail=f"last_auth={last_auth}",
        fail_detail=f"last_auth={last_auth}",
    )
else:
    check(
        name="Authority is authorized for NAAN",
        condition=False,
        ok_detail="",
        fail_detail=f"Authorization did not become true. last_auth={last_auth}",
    )



## 3. Reserve and Resolve ARK (Single)

Creates a RESERVED ARK and verifies retrieval in `R` state.


In [ ]:
reserve_payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "alternate_identifiers": [
        {"schema": "doi", "value": f"10.1234/{uuid.uuid4().hex[:10]}"}
    ],
}

_, reserve_body = call_api(
    name="Reserve single ARK",
    method="POST",
    path="/arks",
    expected=201,
    json=reserve_payload,
)

ark_reserved = reserve_body.get("ark") if isinstance(reserve_body, dict) else None
check(
    name="Reserved ARK ID present",
    condition=bool(ark_reserved),
    ok_detail=f"ark={ark_reserved}",
    fail_detail=f"Unexpected reserve response: {reserve_body}",
)
CONTEXT["ark_reserved"] = ark_reserved

_, get_reserved_body = call_api(
    name="Get reserved ARK",
    method="GET",
    path=f"/arks/{ark_reserved}",
    expected=200,
)

if isinstance(get_reserved_body, dict):
    check(
        name="Reserved ARK state is R",
        condition=get_reserved_body.get("state") == "R",
        ok_detail=str(get_reserved_body),
        fail_detail=str(get_reserved_body),
    )


## 4. Validation and Error Scenarios

Covers malformed identifiers and unauthorized reserve attempts.


In [ ]:
# Invalid ARK format
call_api(
    name="Get ARK with invalid format",
    method="GET",
    path="/arks/not-an-ark",
    expected=400,
)

# Unauthorized reserve attempt
unauthorized_payload = {
    "authority_id": f"{AUTHORITY_ID}-unauthorized",
    "naan": NAAN,
}
call_api(
    name="Reserve ARK unauthorized authority",
    method="POST",
    path="/arks",
    expected=403,
    json=unauthorized_payload,
)


## 5. Update to DRAFT (JSON), Overwrite DRAFT, and Validation Errors

Exercises:

- `RESERVED -> DRAFT`
- `DRAFT -> DRAFT` overwrite
- Empty target validation
- `metadata_format` validation


In [ ]:
ark = CONTEXT.get("ark_reserved")
if not ark:
    skip("Update-to-DRAFT scenarios", "No reserved ARK available from previous step")
else:
    # 5.1 Empty target should fail (target is mandatory)
    empty_target_payload = {
        "authority_id": AUTHORITY_ID,
        "target": "",
        "metadata": json_metadata("invalid-empty-target", 1),
        "metadata_format": "json",
    }
    call_api(
        name="Update RESERVED with empty target",
        method="PUT",
        path=f"/arks/{ark}",
        expected=400,
        json=empty_target_payload,
    )

    # State should remain RESERVED after failed update
    _, after_failed_update = call_api(
        name="State remains RESERVED after invalid update",
        method="GET",
        path=f"/arks/{ark}",
        expected=200,
    )
    if isinstance(after_failed_update, dict):
        check(
            name="ARK still in RESERVED",
            condition=after_failed_update.get("state") == "R",
            ok_detail=str(after_failed_update),
            fail_detail=str(after_failed_update),
        )

    # 5.2 Valid update RESERVED -> DRAFT
    update_payload_v1 = {
        "authority_id": AUTHORITY_ID,
        "target": "https://example.org/resource-v1",
        "metadata": json_metadata("resource-v1", 1),
        "metadata_format": "json",
        "alternate_identifiers": [
            {"schema": "internal", "value": "draft-v1"}
        ],
    }
    _, draft_body_v1 = call_api(
        name="Update RESERVED to DRAFT (json)",
        method="PUT",
        path=f"/arks/{ark}",
        expected=200,
        json=update_payload_v1,
    )
    if isinstance(draft_body_v1, dict):
        check(
            name="ARK state is D after first update",
            condition=draft_body_v1.get("state") == "D",
            ok_detail=str(draft_body_v1),
            fail_detail=str(draft_body_v1),
        )

    # 5.3 Overwrite DRAFT payload (state remains D)
    update_payload_v2 = {
        "authority_id": AUTHORITY_ID,
        "target": "https://example.org/resource-v2",
        "metadata": json_metadata("resource-v2", 2),
        "metadata_format": "json",
        "alternate_identifiers": [
            {"schema": "internal", "value": "draft-v2"}
        ],
    }
    _, draft_body_v2 = call_api(
        name="Overwrite DRAFT payload",
        method="PUT",
        path=f"/arks/{ark}",
        expected=200,
        json=update_payload_v2,
    )
    if isinstance(draft_body_v2, dict):
        check(
            name="ARK remains DRAFT after overwrite",
            condition=draft_body_v2.get("state") == "D",
            ok_detail=str(draft_body_v2),
            fail_detail=str(draft_body_v2),
        )
        check(
            name="Target was overwritten in DRAFT",
            condition=draft_body_v2.get("target") == "https://example.org/resource-v2",
            ok_detail=str(draft_body_v2),
            fail_detail=str(draft_body_v2),
        )

    # 5.4 metadata_format validation (Literal[json, xml])
    invalid_format_payload = {
        "authority_id": AUTHORITY_ID,
        "target": "https://example.org/invalid-format",
        "metadata": "{}",
        "metadata_format": "yaml",
    }
    call_api(
        name="Reject invalid metadata_format",
        method="PUT",
        path=f"/arks/{ark}",
        expected=422,
        json=invalid_format_payload,
    )

    # 5.5 Ownership validation
    wrong_owner_payload = {
        "authority_id": f"{AUTHORITY_ID}-other",
        "target": "https://example.org/owner-check",
        "metadata": json_metadata("owner-check", 1),
        "metadata_format": "json",
    }
    call_api(
        name="Reject update by non-owner authority",
        method="PUT",
        path=f"/arks/{ark}",
        expected=403,
        json=wrong_owner_payload,
    )


## 6. XML Metadata Path

Covers XML payload handling (`metadata_format=xml`).


In [ ]:
# Reserve a second ARK for XML scenario
_, xml_reserve_body = call_api(
    name="Reserve ARK for XML metadata",
    method="POST",
    path="/arks",
    expected=201,
    json={"authority_id": AUTHORITY_ID, "naan": NAAN},
)

ark_xml = xml_reserve_body.get("ark") if isinstance(xml_reserve_body, dict) else None
check(
    name="XML ARK reserve returned ID",
    condition=bool(ark_xml),
    ok_detail=str(ark_xml),
    fail_detail=str(xml_reserve_body),
)

xml_update_payload = {
    "authority_id": AUTHORITY_ID,
    "target": "https://example.org/xml-resource",
    "metadata": xml_metadata("xml-resource"),
    "metadata_format": "xml",
}

_, xml_update_body = call_api(
    name="Update ARK to DRAFT (xml)",
    method="PUT",
    path=f"/arks/{ark_xml}",
    expected=200,
    json=xml_update_payload,
)

if isinstance(xml_update_body, dict):
    check(
        name="XML update leaves ARK in DRAFT",
        condition=xml_update_body.get("state") == "D",
        ok_detail=str(xml_update_body),
        fail_detail=str(xml_update_body),
    )
    check(
        name="XML metadata_format stored",
        condition=xml_update_body.get("metadata_format") == "xml",
        ok_detail=str(xml_update_body),
        fail_detail=str(xml_update_body),
    )

CONTEXT["ark_xml"] = ark_xml


## 7. Checkdigit and ARK Parsing Behavior

Mutates the reserved ARK suffix to verify invalid checkdigit handling.
If checkdigit validation is disabled, this may return `404` instead of `400`.


In [ ]:
valid_ark = CONTEXT.get("ark_reserved")
if not valid_ark:
    skip("Checkdigit mutation scenario", "No reserved ARK available from previous step")
else:
    naan_part, name_part = valid_ark[4:].split("/", 1)

    # Flip last character to force an invalid name/checkdigit candidate.
    replacement = "0" if name_part[-1] != "0" else "1"
    mutated_name = name_part[:-1] + replacement
    invalid_ark = f"ark:{naan_part}/{mutated_name}"

    resp, body = call_api(
        name="Get ARK with mutated checkdigit/name",
        method="GET",
        path=f"/arks/{invalid_ark}",
        expected=[400, 404],
    )

    if resp is not None:
        check(
            name="Mutated ARK rejected as expected",
            condition=resp.status_code in {400, 404},
            ok_detail=f"status={resp.status_code}",
            fail_detail=f"status={resp.status_code}, body={body}",
        )


## 8. Tombstone Lifecycle

Covers `DELETE` semantics and idempotency.

- `DRAFT -> TOMBSTONE`
- `TOMBSTONE -> TOMBSTONE` on repeated delete
- update blocked on tombstoned record


In [ ]:
ark = CONTEXT.get("ark_reserved")
if not ark:
    skip("Tombstone lifecycle", "No reserved ARK available from previous step")
else:
    call_api(
        name="Delete ARK (to tombstone)",
        method="DELETE",
        path=f"/arks/{ark}",
        expected=[200, 204],
    )

    _, tombstone_get_body = call_api(
        name="Get tombstoned ARK",
        method="GET",
        path=f"/arks/{ark}",
        expected=200,
    )
    if isinstance(tombstone_get_body, dict):
        check(
            name="ARK state is TOMBSTONE",
            condition=tombstone_get_body.get("state") == "T",
            ok_detail=str(tombstone_get_body),
            fail_detail=str(tombstone_get_body),
        )

    call_api(
        name="Delete tombstoned ARK (idempotent)",
        method="DELETE",
        path=f"/arks/{ark}",
        expected=[200, 204],
    )

    blocked_update_payload = {
        "authority_id": AUTHORITY_ID,
        "target": "https://example.org/should-not-update",
        "metadata": json_metadata("blocked-update", 1),
        "metadata_format": "json",
    }
    call_api(
        name="Reject update on tombstoned ARK",
        method="PUT",
        path=f"/arks/{ark}",
        expected=409,
        json=blocked_update_payload,
    )


## 9. Batch Reserve Scenarios

Covers successful batch mint and payload validation failures.


In [ ]:
batch_payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "items": [
        {
            "client_item_id": "item-1",
            "target": "https://example.org/batch/1",
            "alternate_identifiers": [{"schema": "idx", "value": "1"}],
        },
        {
            "client_item_id": "item-2",
            "target": "https://example.org/batch/2",
        },
    ],
}

_, batch_body = call_api(
    name="Batch reserve success",
    method="POST",
    path="/arks/batch",
    expected=200,
    json=batch_payload,
)

if isinstance(batch_body, dict):
    results = batch_body.get("results") or []
    errors = batch_body.get("errors")
    check(
        name="Batch returns 2 results",
        condition=len(results) == 2,
        ok_detail=f"results={len(results)}",
        fail_detail=str(batch_body),
    )
    check(
        name="Batch success has no errors",
        condition=not errors,
        ok_detail=str(batch_body),
        fail_detail=str(batch_body),
    )

# Validation: missing client_item_id in one item -> 422
invalid_batch_payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "items": [
        {"target": "https://example.org/batch-invalid/no-client-id"}
    ],
}
call_api(
    name="Batch validation rejects missing client_item_id",
    method="POST",
    path="/arks/batch",
    expected=422,
    json=invalid_batch_payload,
)


## 10. Concurrency Scenario (Parallel Reserve)

Launches concurrent reserve requests and verifies uniqueness of minted ARKs.
This is a practical integration check for counter allocation under concurrency.


In [ ]:
def reserve_once(i: int):
    payload = {
        "authority_id": AUTHORITY_ID,
        "naan": NAAN,
        "alternate_identifiers": [{"schema": "bulk", "value": str(i)}],
    }
    try:
        r = requests.post(f"{MINTER_API_V1}/arks", json=payload, timeout=30)
        body = _decode_body(r)
        return r.status_code, body
    except Exception as exc:
        return None, str(exc)

parallel_n = 20
statuses = []
arks = []
errors = []

with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(reserve_once, i) for i in range(parallel_n)]
    for fut in as_completed(futures):
        status, body = fut.result()
        statuses.append(status)
        if status == 201 and isinstance(body, dict) and body.get("ark"):
            arks.append(body["ark"])
        else:
            errors.append({"status": status, "body": body})

check(
    name="All parallel reserve requests returned 201",
    condition=len(errors) == 0 and len(statuses) == parallel_n,
    ok_detail=f"count={len(statuses)}",
    fail_detail=f"errors={errors}",
)

check(
    name="Parallel reserve ARKs are unique",
    condition=len(arks) == len(set(arks)) == parallel_n,
    ok_detail=f"unique={len(set(arks))}",
    fail_detail=f"received={len(arks)} unique={len(set(arks))}",
)


## 11. Worker Publish and Update Flow (Optional)

If worker is running, this section validates:

- `DRAFT -> PUBLISHED` (worker create)
- `PUBLISHED -> UPDATE` (`PUT`)
- `UPDATE -> UPDATE` overwrite (`PUT`)
- `UPDATE -> PUBLISHED` (worker update)

If worker is not running, checks are marked as skipped.


In [ ]:
if not CONTEXT.get("worker_running"):
    skip("Worker flow", "Worker is not running according to /worker/status")
else:
    # Reserve and move to DRAFT
    _, worker_reserve_body = call_api(
        name="Worker flow reserve",
        method="POST",
        path="/arks",
        expected=201,
        json={"authority_id": AUTHORITY_ID, "naan": NAAN},
    )
    ark_worker = worker_reserve_body.get("ark") if isinstance(worker_reserve_body, dict) else None

    if not ark_worker:
        record("Worker flow reserve returned ARK", "FAIL", str(worker_reserve_body))
    else:
        call_api(
            name="Worker flow set DRAFT",
            method="PUT",
            path=f"/arks/{ark_worker}",
            expected=200,
            json={
                "authority_id": AUTHORITY_ID,
                "target": "https://example.org/worker-create",
                "metadata": json_metadata("worker-create", 1),
                "metadata_format": "json",
            },
        )

        ok_published, published_body = wait_for_state(ark_worker, expected_state="P")
        check(
            name="Worker transitions DRAFT to PUBLISHED",
            condition=ok_published,
            ok_detail=str(published_body),
            fail_detail=f"Timed out waiting for PUBLISHED. Last body={published_body}",
        )

        # PUBLISHED -> UPDATE
        _, update_body = call_api(
            name="PUT on published ARK transitions to UPDATE",
            method="PUT",
            path=f"/arks/{ark_worker}",
            expected=200,
            json={
                "authority_id": AUTHORITY_ID,
                "target": "https://example.org/worker-update-v1",
                "metadata": json_metadata("worker-update-v1", 2),
                "metadata_format": "json",
            },
        )
        if isinstance(update_body, dict):
            check(
                name="Published ARK moved to UPDATE",
                condition=update_body.get("state") == "U",
                ok_detail=str(update_body),
                fail_detail=str(update_body),
            )

        # UPDATE -> UPDATE overwrite
        _, update_overwrite_body = call_api(
            name="Overwrite pending UPDATE",
            method="PUT",
            path=f"/arks/{ark_worker}",
            expected=200,
            json={
                "authority_id": AUTHORITY_ID,
                "target": "https://example.org/worker-update-v2",
                "metadata": json_metadata("worker-update-v2", 3),
                "metadata_format": "json",
            },
        )
        if isinstance(update_overwrite_body, dict):
            check(
                name="ARK remains UPDATE after overwrite",
                condition=update_overwrite_body.get("state") == "U",
                ok_detail=str(update_overwrite_body),
                fail_detail=str(update_overwrite_body),
            )

        ok_republished, republished_body = wait_for_state(ark_worker, expected_state="P")
        check(
            name="Worker transitions UPDATE back to PUBLISHED",
            condition=ok_republished,
            ok_detail=str(republished_body),
            fail_detail=f"Timed out waiting for republish. Last body={republished_body}",
        )


## 12. Optional: Chain Import Path (`missing local -> on-chain exists -> UPDATE`)

Set `EXISTING_CHAIN_ARK` to an ARK that already exists on-chain but is absent locally.
The API should import it as local `PUBLISHED` and then transition it to `UPDATE` in the same `PUT` call.


In [ ]:
if not EXISTING_CHAIN_ARK:
    skip("Chain import scenario", "EXISTING_CHAIN_ARK is not set")
else:
    _, import_update_body = call_api(
        name="PUT imports on-chain ARK then sets UPDATE",
        method="PUT",
        path=f"/arks/{EXISTING_CHAIN_ARK}",
        expected=[200, 404],
        json={
            "authority_id": AUTHORITY_ID,
            "target": "https://example.org/import-update",
            "metadata": json_metadata("import-update", 1),
            "metadata_format": "json",
        },
    )

    if isinstance(import_update_body, dict) and "state" in import_update_body:
        check(
            name="Imported ARK ends in UPDATE state",
            condition=import_update_body.get("state") == "U",
            ok_detail=str(import_update_body),
            fail_detail=str(import_update_body),
        )


## 13. Missing Record Update/Delete Checks

Verifies behavior for operations on non-existing ARKs.


In [ ]:
missing_ark = f"ark:{NAAN}/does-not-exist-{uuid.uuid4().hex[:8]}"

call_api(
    name="Update missing ARK returns 404",
    method="PUT",
    path=f"/arks/{missing_ark}",
    expected=[400, 404],
    json={
        "authority_id": AUTHORITY_ID,
        "target": "https://example.org/missing",
        "metadata": json_metadata("missing-update", 1),
        "metadata_format": "json",
    },
)

call_api(
    name="Delete missing ARK returns 404",
    method="DELETE",
    path=f"/arks/{missing_ark}",
    expected=[400, 404],
)



## 14. Final Summary

This cell prints a consolidated pass/fail summary and raises an exception if any check failed.


In [ ]:
summary = summarize_results()

if summary["failed"] > 0:
    raise AssertionError(f"There are {summary['failed']} failed checks. Review notebook output.")
else:
    print("All checked scenarios passed.")
